# LLM04: Tokenization, Input Embedding & Positional Encoding

## Lab Overview

This lab covers the first stage of the LLM pipeline: converting raw text into the numerical representations that a Transformer model consumes. You will explore tokenization methods, understand how token IDs are mapped to dense embedding vectors, and implement positional encoding schemes — including the **Rotary Position Embedding (RoPE)** used in LLaMA.

> **Quick terms:** A **token** (or **token ID**) is an integer index into the model vocabulary. An **embedding** maps each token ID to a dense vector. **Attention masks** (`attention_mask`) flag real tokens vs. padding; the **attention** mechanism itself (how vectors mix) is built in LLM06.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

Gain hands-on understanding of how text enters a Large Language Model, from sub-word tokenization through embedding lookup to position-aware representations.

1. **Tokenize Text**: Use HuggingFace tokenizers (`tokenize`, `encode`, `decode`, batch processing).
2. **Understand Embedding Lookup**: See how `nn.Embedding` maps discrete token IDs to continuous vectors.
3. **Implement Sinusoidal Positional Encoding**: Code Rotary Position Embedding and verify that the rotated dot product depends on relative position.
4. **Implement RoPE**: Code Rotary Position Embedding and verify its distance-decay property.
5. **Master Tensor Operations**: Practice `view`, `reshape`, `transpose`, `permute`, broadcasting, `gather`, and `scatter`.

---


## 1. Environment Setup


In [26]:
# Core libraries
import math
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

Using device: cuda
PyTorch version: 2.9.1+rocm7.2.0.git7e1940d4
GPU: AMD Radeon(TM) 8060S Graphics
GPU Memory: 102.7 GB


## 2. Tokenization with HuggingFace

Tokenization converts raw text into a sequence of sub-word tokens. Modern LLMs commonly use subword tokenizers built with algorithms such as **BPE** or **Unigram**, often implemented through libraries like **SentencePiece**.

**Key Tokenizer Methods:**

- `tokenize(text)` — split text into sub-word string tokens (no special tokens)
- `encode(text)` — convert text to a list of integer token IDs
- `decode(ids)` — convert token IDs back to human-readable text
- `tokenizer(texts, ...)` — full pipeline returning `input_ids` and `attention_mask` (supports batching & padding)


In [27]:
from transformers import AutoTokenizer

# Load a LLaMA-compatible tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "Nanjing University is a prestigious university"
print(f"Original text: {text}")

# --- tokenize(): sub-word splitting only ---
# tokenize() returns token strings only
tokens = tokenizer.tokenize(text)
print(f"\ntokenize() -> {tokens}")
print(f"Number of tokens: {len(tokens)}")

# --- encode(): text -> token IDs ---
# encode() returns token IDs
ids_no_special = tokenizer.encode(text, add_special_tokens=False)
ids_with_special = tokenizer.encode(text, add_special_tokens=True)
print(f"\nencode(no special):  {ids_no_special}")
print(f"encode(with special): {ids_with_special}")

# --- decode(): token IDs -> text ---
decoded = tokenizer.decode(ids_with_special)
print(f"\ndecode() -> '{decoded}'")

# --- Full pipeline: tokenizer() ---
# tokenizer(...) is the full preprocessing pipeline and is the most commonly used interface in model code
encoded = tokenizer(text)
print(f"\ntokenizer() output keys: {list(encoded.keys())}")
print(f"input_ids:      {encoded['input_ids']}")
print(f"attention_mask: {encoded['attention_mask']}")

Original text: Nanjing University is a prestigious university

tokenize() -> ['▁Nan', 'j', 'ing', '▁University', '▁is', '▁a', '▁pr', 'estig', 'ious', '▁university']
Number of tokens: 10

encode(no special):  [25701, 29926, 292, 3014, 338, 263, 544, 5286, 2738, 16372]
encode(with special): [1, 25701, 29926, 292, 3014, 338, 263, 544, 5286, 2738, 16372]

decode() -> '<s> Nanjing University is a prestigious university'

tokenizer() output keys: ['input_ids', 'attention_mask']
input_ids:      [1, 25701, 29926, 292, 3014, 338, 263, 544, 5286, 2738, 16372]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [28]:
# Special tokens inspection
print("=== Special Token Info ===")
print(f"BOS token: {tokenizer.bos_token!r}  (ID: {tokenizer.bos_token_id})")
print(f"EOS token: {tokenizer.eos_token!r}  (ID: {tokenizer.eos_token_id})")
print(f"PAD token: {tokenizer.pad_token!r}  (ID: {tokenizer.pad_token_id})")
print(f"UNK token: {tokenizer.unk_token!r}  (ID: {tokenizer.unk_token_id})")
print(f"Vocabulary size: {tokenizer.vocab_size}")
# Many causal LLM tokenizers do not define a native PAD token.
# For batching, it is common to reuse EOS as PAD during inference demos.

=== Special Token Info ===
BOS token: '<s>'  (ID: 1)
EOS token: '</s>'  (ID: 2)
PAD token: '</s>'  (ID: 2)
UNK token: '<unk>'  (ID: 0)
Vocabulary size: 32000


In [29]:
# Batch tokenization with padding
texts = ["Hello world", "Nanjing University", "Artificial Intelligence and LLMs"]
print(f"Batch texts: {texts}")

# For simple batching demos, reuse EOS as PAD if the tokenizer has no PAD token.
# Real training/inference pipelines may handle padding more carefully.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

batch_encoded = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
print(f"\ninput_ids shape:      {batch_encoded['input_ids'].shape}")
print(f"attention_mask shape: {batch_encoded['attention_mask'].shape}")
print(f"\ninput_ids:\n{batch_encoded['input_ids']}")
print(f"\nattention_mask:\n{batch_encoded['attention_mask']}")

# Decode back
for i, ids in enumerate(batch_encoded["input_ids"]):
    print(f"  [{i}] {tokenizer.decode(ids, skip_special_tokens=True)}")

Batch texts: ['Hello world', 'Nanjing University', 'Artificial Intelligence and LLMs']

input_ids shape:      torch.Size([3, 10])
attention_mask shape: torch.Size([3, 10])

input_ids:
tensor([[    1, 15043,  3186,     2,     2,     2,     2,     2,     2,     2],
        [    1, 25701, 29926,   292,  3014,     2,     2,     2,     2,     2],
        [    1,  3012,   928,   616,  3159, 28286,   322,   365, 26369, 29879]])

attention_mask:
tensor([[1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
  [0] Hello world
  [1] Nanjing University
  [2] Artificial Intelligence and LLMs


## 3. Input Embedding — From Token IDs to Vectors

After tokenization each token is represented by an integer ID. The **embedding layer** (`nn.Embedding`) maps each ID to a learnable dense vector of dimension `hidden_dim`.

$$\mathbf{e}_i = \text{EmbeddingTable}[\text{token\_id}_i] \in \mathbb{R}^{d}$$

For modern LLMs, the embedding table can be very large because its parameter count is:

$$
\text{vocab\_size} \times \text{hidden\_dim}
$$

For example, a model with vocabulary size 128,256 and hidden size 2,048 would need about 262 million embedding parameters in its embedding table alone.


In [30]:
# Demonstrate nn.Embedding
vocab_size = 32000  # toy vocabulary size for demonstration; tokenizer.vocab_size may differ due to special tokens
hidden_dim = 128  # toy hidden size for demonstration

embedding = nn.Embedding(vocab_size, hidden_dim).to(device)
print(f"Embedding table shape: {embedding.weight.shape}  (vocab_size x hidden_dim)")
print(f"Parameters: {embedding.weight.numel():,}")

# Look up embeddings for a short sentence
sample_ids = torch.tensor([[1, 4123, 567, 89, 2]], device=device)  # (batch=1, seq_len=5)
embedded = embedding(sample_ids)
print(f"\nInput IDs shape:  {sample_ids.shape}")
print(f"Embedded shape:   {embedded.shape}   # (batch, seq_len, hidden_dim)")

# Verify: the embedding for token 4123 equals row 4123 of the weight matrix
assert torch.allclose(embedded[0, 1], embedding.weight[4123])
print("Verified: embedded[0,1] == embedding.weight[4123]  ✓")

Embedding table shape: torch.Size([32000, 128])  (vocab_size x hidden_dim)
Parameters: 4,096,000

Input IDs shape:  torch.Size([1, 5])
Embedded shape:   torch.Size([1, 5, 128])   # (batch, seq_len, hidden_dim)
Verified: embedded[0,1] == embedding.weight[4123]  ✓


## 4. Sinusoidal Positional Encoding (Vaswani et al., 2017)

Transformers have no built-in notion of token order. The original _Attention Is All You Need_ paper adds a **fixed** positional encoding:

$$
PE_{(pos, 2i)}   = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \qquad
  PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)
$$

**Properties:**

- Each position gets a deterministic encoding.
- Nearby positions often have more similar encodings than very distant positions.
- Because the encoding uses sin/cos pairs at multiple frequencies, interactions between positions can reflect relative offsets.


In [31]:
def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of sinusoidal positional encodings."""
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (max_len, 1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


max_len, d_model = 64, 128
pe = sinusoidal_positional_encoding(max_len, d_model)
print(f"Positional encoding shape: {pe.shape}")

# Example: compare positional similarity at two different offsets
dot_01 = torch.dot(pe[0], pe[1]).item()
dot_0_31 = torch.dot(pe[0], pe[31]).item()
print(f"dot(PE[0], PE[1])  = {dot_01:.4f}")
print(f"dot(PE[0], PE[31]) = {dot_0_31:.4f}")
print("Different relative offsets produce different similarities.")

Positional encoding shape: torch.Size([64, 128])
dot(PE[0], PE[1])  = 62.0937
dot(PE[0], PE[31]) = 36.2908
Different relative offsets produce different similarities.


In [32]:
# Visualize sinusoidal positional encoding (text-based heatmap summary)
print("=== PE value ranges across dimensions ===")
for dim_idx in [0, 1, 10, 50, 126, 127]:
    col = pe[:, dim_idx]
    print(f"  dim {dim_idx:>3d}: min={col.min():.4f}, max={col.max():.4f}, type={'sin' if dim_idx % 2 == 0 else 'cos'}")

# Show that adding PE to embeddings preserves shape
sample_embed = torch.randn(1, max_len, d_model)
positioned = sample_embed + pe.unsqueeze(0)  # broadcasting: (1, max_len, d_model)
print(f"\nEmbedding + PE shape: {positioned.shape}")

=== PE value ranges across dimensions ===
  dim   0: min=-1.0000, max=0.9999, type=sin
  dim   1: min=-1.0000, max=1.0000, type=cos
  dim  10: min=-0.9902, max=0.9999, type=sin
  dim  50: min=0.0000, max=1.0000, type=sin
  dim 126: min=0.0000, max=0.0073, type=sin
  dim 127: min=1.0000, max=1.0000, type=cos

Embedding + PE shape: torch.Size([1, 64, 128])


## 5. Rotary Position Embedding (RoPE)

Modern LLMs like **LLaMA** replace additive sinusoidal PE with **Rotary Position Embedding (RoPE)** (Su et al., 2021). Instead of _adding_ position information, RoPE _rotates_ query and key vectors.

**Core Idea (2-D case):**
For a 2-D vector $\mathbf{x} = (x_0, x_1)$ at position $m$:

$$R_m \mathbf{x} = \begin{pmatrix} \cos m\theta & -\sin m\theta \\ \sin m\theta & \cos m\theta \end{pmatrix} \begin{pmatrix} x_0 \\ x_1 \end{pmatrix}$$

**Key property:** $\langle R_m \mathbf{q}, R_n \mathbf{k} \rangle = \langle R_{m-n} \mathbf{q}, \mathbf{k} \rangle$ — the dot product depends only on the _relative_ position offset $m - n$.

**n-D generalization:** Pair up dimensions $(x_0, x_1), (x_2, x_3), \ldots$ and rotate each pair with a different frequency $\theta_i = 10000^{-2i/d}$.

**Implementation formula (element-wise):**
In practice, RoPE splits the hidden dimension into consecutive pairs and applies a 2D rotation to each pair.

$$\text{even}' = x_{\text{even}} \cos\theta - x_{\text{odd}} \sin\theta$$
$$\text{odd}' = x_{\text{odd}} \cos\theta + x_{\text{even}} \sin\theta$$


In [33]:
def build_rope_cache(seq_len: int, dim: int, base: float = 10000.0, device: torch.device = None) -> tuple:
    """Pre-compute cos and sin tables for RoPE.

    Returns:
        cos, sin — each of shape (seq_len, dim // 2)
    """
    assert dim % 2 == 0, "RoPE requires an even hidden dimension."
    device = device or torch.device("cpu")
    position = torch.arange(seq_len, dtype=torch.float32, device=device)
    dim_idx = torch.arange(dim // 2, dtype=torch.float32, device=device)
    inv_freq = base ** (-dim_idx / (dim // 2))
    freqs = torch.outer(position, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply Rotary Position Embedding to tensor x.

    Args:
        x:   (batch, seq_len, dim)
        cos: (seq_len, dim // 2)
        sin: (seq_len, dim // 2)
    Returns:
        Rotated tensor of same shape as x.
    """
    assert x.shape[-1] % 2 == 0, "RoPE requires an even hidden dimension."
    cos = cos.unsqueeze(0)
    sin = sin.unsqueeze(0)
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    rotated_even = x_even * cos - x_odd * sin
    rotated_odd = x_odd * cos + x_even * sin
    return torch.stack([rotated_even, rotated_odd], dim=-1).flatten(-2)


# --- Quick test ---
seq_len, dim = 4, 8
assert dim % 2 == 0, "RoPE requires an even hidden dimension."
hidden = torch.arange(seq_len * dim, dtype=torch.float32).view(1, seq_len, dim)
cos, sin = build_rope_cache(seq_len, dim)
rotated = apply_rope(hidden, cos, sin)

print(f"Input shape:   {hidden.shape}")
print(f"cos/sin shape: {cos.shape}")
print(f"Output shape:  {rotated.shape}")
print(f"\nOriginal (position 0): {hidden[0, 0]}")
print(f"After RoPE (pos 0):    {rotated[0, 0]}")

Input shape:   torch.Size([1, 4, 8])
cos/sin shape: torch.Size([4, 4])
Output shape:  torch.Size([1, 4, 8])

Original (position 0): tensor([0., 1., 2., 3., 4., 5., 6., 7.])
After RoPE (pos 0):    tensor([0., 1., 2., 3., 4., 5., 6., 7.])


In [34]:
# Verify RoPE's relative-position property more carefully.
# If the underlying q and k vectors are the same across all positions,
# then <R_m q, R_n k> depends only on the relative offset (m - n).

seq_len, dim = 16, 64
cos, sin = build_rope_cache(seq_len, dim)

# Use one shared q and one shared k, then copy them to every position
q_base = torch.randn(1, 1, dim)
k_base = torch.randn(1, 1, dim)

q_same = q_base.expand(1, seq_len, dim).clone()
k_same = k_base.expand(1, seq_len, dim).clone()

q_rot = apply_rope(q_same, cos, sin)
k_rot = apply_rope(k_same, cos, sin)


def dot_at(qr, kr, m, n):
    return torch.dot(qr[0, m], kr[0, n]).item()


print("=== Verifying relative-position dependence ===")
for offset in [0, 1, 3, 5]:
    values = [dot_at(q_rot, k_rot, m, m + offset) for m in range(seq_len - offset)]
    max_diff = max(values) - min(values)
    print(f"offset={offset:>2d}: first={values[0]:+.6f}, last={values[-1]:+.6f}, max_diff={max_diff:.6e}")

print("\nIf max_diff is near zero, the dot product is determined by the relative offset.")

=== Verifying relative-position dependence ===
offset= 0: first=+0.778705, last=+0.778705, max_diff=1.907349e-06
offset= 1: first=+0.176626, last=+0.176628, max_diff=2.861023e-06
offset= 3: first=-2.739197, last=-2.739194, max_diff=4.768372e-06
offset= 5: first=-1.254009, last=-1.254008, max_diff=1.907349e-06

If max_diff is near zero, the dot product is determined by the relative offset.


## 6. Essential Tensor Operations

LLM implementations heavily use tensor reshaping, transposition, and advanced indexing. Below we practice the most important operations and connect them to real Transformer code.

### 6.1 `view` vs `reshape`

- `view` requires the tensor to be **contiguous** in memory
- `reshape` works on any tensor (may copy data if needed)


In [35]:
x = torch.randn(2, 3, 4)
print(f"Original shape: {x.shape}, contiguous: {x.is_contiguous()}")

# view and reshape behave the same on contiguous tensors
y_view = x.view(6, 4)
y_reshape = x.reshape(6, 4)
print(f"view(6,4):    {y_view.shape}")
print(f"reshape(6,4): {y_reshape.shape}")

# After transpose the tensor is NOT contiguous
x_t = x.transpose(0, 1)  # (3, 2, 4)
print(f"\nAfter transpose: shape={x_t.shape}, contiguous={x_t.is_contiguous()}")

# view fails on non-contiguous tensors
try:
    x_t.view(6, 4)
except RuntimeError as e:
    print(f"view() error: {e}")

# reshape handles it automatically
y_t = x_t.reshape(6, 4)
print(f"reshape() succeeds: {y_t.shape}")

Original shape: torch.Size([2, 3, 4]), contiguous: True
view(6,4):    torch.Size([6, 4])
reshape(6,4): torch.Size([6, 4])

After transpose: shape=torch.Size([3, 2, 4]), contiguous=False
view() error: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
reshape() succeeds: torch.Size([6, 4])


### 6.2 `transpose` vs `permute`

- `transpose(dim0, dim1)` swaps exactly **two** dimensions
- `permute(dims)` can reorder **all** dimensions at once

In multi-head attention, `permute` is used to rearrange `(batch, seq, num_heads, head_dim)` → `(batch, num_heads, seq, head_dim)`.


In [36]:
x = torch.randn(2, 3, 4, 5)
print(f"Original: {x.shape}")

# transpose swaps two dims
y1 = x.transpose(1, 3)  # (2, 5, 4, 3)
print(f"transpose(1,3): {y1.shape}")

# permute reorders all dims
y2 = x.permute(0, 3, 1, 2)  # (2, 5, 3, 4)
print(f"permute(0,3,1,2): {y2.shape}")

# Common MHA layout change:
# (batch, seq, num_heads, head_dim) -> (batch, num_heads, seq, head_dim)
batch, seq, num_heads, head_dim = 2, 10, 8, 64
q = torch.randn(batch, seq, num_heads, head_dim)
q_mha = q.permute(0, 2, 1, 3)  # standard MHA layout
print(f"\nMHA reshape: {q.shape} -> {q_mha.shape}")

Original: torch.Size([2, 3, 4, 5])
transpose(1,3): torch.Size([2, 5, 4, 3])
permute(0,3,1,2): torch.Size([2, 5, 3, 4])

MHA reshape: torch.Size([2, 10, 8, 64]) -> torch.Size([2, 8, 10, 64])


### 6.3 `squeeze` & `unsqueeze`


In [37]:
x = torch.randn(1, 3, 1, 4)
print(f"Original: {x.shape}")
print(f"squeeze():   {x.squeeze().shape}")  # remove ALL size-1 dims
print(f"squeeze(0):  {x.squeeze(0).shape}")  # remove dim 0 only
print(f"squeeze(2):  {x.squeeze(2).shape}")  # remove dim 2 only

y = torch.randn(3, 4)
print(f"\nOriginal: {y.shape}")
print(f"unsqueeze(0):  {y.unsqueeze(0).shape}")  # add batch dim
print(f"unsqueeze(-1): {y.unsqueeze(-1).shape}")  # add trailing dim
print(f"unsqueeze(1):  {y.unsqueeze(1).shape}")  # add middle dim

Original: torch.Size([1, 3, 1, 4])
squeeze():   torch.Size([3, 4])
squeeze(0):  torch.Size([3, 1, 4])
squeeze(2):  torch.Size([1, 3, 4])

Original: torch.Size([3, 4])
unsqueeze(0):  torch.Size([1, 3, 4])
unsqueeze(-1): torch.Size([3, 4, 1])
unsqueeze(1):  torch.Size([3, 1, 4])


### 6.4 Broadcasting

Broadcasting allows PyTorch to automatically expand tensors with compatible shapes, so we can apply element-wise operations without manually copying data.

**Rules** (NumPy-compatible, compared from right to left):

1. If one tensor has fewer dimensions, prepend size-1 dimensions to its shape.
2. Two dimensions are compatible if they are **equal** or if one of them is **1**.
3. If any compared dimensions are incompatible, PyTorch raises a `RuntimeError`.

| A shape        | B shape        | Result shape   | Explanation                                       |
| -------------- | -------------- | -------------- | ------------------------------------------------- |
| `(4,)`         | scalar         | `(4,)`         | Scalar broadcasts to every element                |
| `(3, 4)`       | `(4,)`         | `(3, 4)`       | Row-wise broadcast across the first dimension     |
| `(3, 4)`       | `(3, 1)`       | `(3, 4)`       | Column-wise broadcast across the second dimension |
| `(2, 8, 8)`    | `(1, 1, 8)`    | `(2, 8, 8)`    | Simple mask broadcast across batch and rows       |
| `(2, 4, 8, 8)` | `(1, 1, 8, 8)` | `(2, 4, 8, 8)` | Additive attention mask broadcast                 |
| `(2, 4, 8, 8)` | `(1, 4, 8, 8)` | `(2, 4, 8, 8)` | Head-specific position bias broadcast             |
| `(3, 4)`       | `(5,)`         | **ERROR**      | Incompatible dimensions                           |

**Common Transformer examples include:**

- Adding a bias vector to token embeddings: `(B, S, D) + (D,)`
- Applying a simple mask across one dimension
- Applying an additive attention mask: `(B, H, S, S) + (1, 1, S, S)`
- Adding head-specific position bias: `(B, H, S, S) + (1, H, S, S)`

This lets us write compact Transformer code while relying on PyTorch to align shapes automatically.


In [38]:
# ---- Level 1: Scalar broadcast ----
a = torch.tensor([1.0, 2.0, 3.0, 4.0])
result = a + 2.0  # scalar -> (4,)
print(f"1) Scalar broadcast:  (4,) + scalar      -> {result.shape}")

# ---- Level 2: Vector broadcast (row direction) ----
A = torch.randn(3, 4)
b = torch.randn(4)  # (4,) -> treated as (1, 4) -> expand to (3, 4)
result = A + b
print(f"2) Vector broadcast:  (3,4) + (4,)       -> {result.shape}")

# ---- Level 3: Column broadcast ----
col = torch.randn(3, 1)  # (3,1) -> expand to (3,4)
result = A + col
print(f"3) Column broadcast:  (3,4) + (3,1)      -> {result.shape}")

# ---- Level 4: Simple mask broadcast ----
# A simple example: broadcast a 1D keep/drop mask across the last dimension.
# This demonstrates the broadcasting rule clearly, even though real attention
# masking is often done with additive masks rather than elementwise multiplication.
B, S = 2, 8
scores = torch.randn(B, S, S)
mask_1d = torch.tensor([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0])  # (S,)
masked = scores * mask_1d.unsqueeze(0).unsqueeze(0)  # (1,1,S) -> (B,S,S)
print(f"4) Simple mask:       ({B},{S},{S}) * (1,1,{S})   -> {masked.shape}")

# ---- Level 5: Transformer-style additive attention mask ----
# In real Transformers, masking is often additive:
# attention scores (B, H, S, S) + mask (1, 1, S, S)
H = 4
scores_attn = torch.randn(B, H, S, S)

# Causal mask: block attention to future positions
causal_mask = torch.triu(torch.full((S, S), float("-inf")), diagonal=1)  # (S,S)
masked_scores_attn = scores_attn + causal_mask.unsqueeze(0).unsqueeze(0)  # (1,1,S,S) -> (B,H,S,S)

print(f"5) Additive mask:     ({B},{H},{S},{S}) + (1,1,{S},{S}) -> {masked_scores_attn.shape}")
print(f"   Causal mask row 0: {causal_mask[0, :4].tolist()}")

# ---- Level 6: Attention + Bias (ALiBi / position bias) ----
# Another common pattern: scores (B, H, S, S) + bias (1, H, S, S)
# Each head gets a different distance-based bias.
scores_4d = torch.randn(B, H, S, S)

positions = torch.arange(S, dtype=torch.float32)
distance_matrix = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()  # (S, S)

head_slopes = torch.tensor([0.5, 0.25, 0.125, 0.0625]).view(H, 1, 1)  # (H,1,1)
bias = -(head_slopes * distance_matrix)  # (H,S,S)
bias = bias.unsqueeze(0)  # (1,H,S,S) -> broadcast over batch

biased_scores = scores_4d + bias
print(f"6) Attn + bias:       ({B},{H},{S},{S}) + (1,{H},{S},{S}) -> {biased_scores.shape}")
print(f"   Bias sample (head 0, pos 0->0..3): {bias[0, 0, 0, :4].tolist()}")

# ---- Level 7: Broadcasting failure ----
try:
    _ = torch.randn(3, 4) + torch.randn(5)
except RuntimeError as e:
    print(f"7) Expected failure:  (3,4) + (5,)       -> {e}")

1) Scalar broadcast:  (4,) + scalar      -> torch.Size([4])
2) Vector broadcast:  (3,4) + (4,)       -> torch.Size([3, 4])
3) Column broadcast:  (3,4) + (3,1)      -> torch.Size([3, 4])
4) Simple mask:       (2,8,8) * (1,1,8)   -> torch.Size([2, 8, 8])
5) Additive mask:     (2,4,8,8) + (1,1,8,8) -> torch.Size([2, 4, 8, 8])
   Causal mask row 0: [0.0, -inf, -inf, -inf]
6) Attn + bias:       (2,4,8,8) + (1,4,8,8) -> torch.Size([2, 4, 8, 8])
   Bias sample (head 0, pos 0->0..3): [-0.0, -0.5, -1.0, -1.5]
7) Expected failure:  (3,4) + (5,)       -> The size of tensor a (4) must match the size of tensor b (5) at non-singleton dimension 1


### 6.5 `gather` & `scatter`

`gather` selects values from a tensor according to indices. `scatter` writes values back into selected positions, which is conceptually related but not always a strict inverse.


In [39]:
# gather: select one element per row according to an index
x = torch.randn(3, 4)
indices = torch.tensor([[0], [2], [1]])  # one index per row
gathered = torch.gather(x, dim=1, index=indices)

print(f"Source:\n{x}")
print(f"Indices: {indices.squeeze().tolist()}")
print(f"Gathered: {gathered.squeeze().tolist()}")
# Verify by hand
for row in range(3):
    col = indices[row, 0].item()
    assert gathered[row, 0] == x[row, col]
print("Gather verification passed ✓")

# scatter: write values into selected positions
target = torch.zeros(3, 4)
values = torch.randn(3, 2)
idx = torch.tensor([[0, 2], [1, 3], [0, 1]])
target.scatter_(1, idx, values)
print(f"\nScatter result:\n{target}")

Source:
tensor([[ 0.3367,  0.2872,  0.6090,  1.0300],
        [ 0.1702, -0.8270,  0.3508, -1.3557],
        [-1.0644, -0.1838,  1.7108, -1.1666]])
Indices: [0, 2, 1]
Gathered: [0.3366703689098358, 0.3507833480834961, -0.18379847705364227]
Gather verification passed ✓

Scatter result:
tensor([[ 0.9718,  0.0000, -0.9686,  0.0000],
        [ 0.0000, -0.3775,  0.0000,  2.6201],
        [-0.3095, -0.0209,  0.0000,  0.0000]])


## Conclusions

### Technical Concepts Learned

- **Tokenization**: Converting text into subword tokens, and using common tokenizer interfaces such as `tokenize`, `encode`, `decode`, and batch tokenization with padding
- **Input Embedding**: Using `nn.Embedding` to map discrete token IDs to dense learnable vectors
- **Sinusoidal Positional Encoding**: Adding fixed position-dependent signals built from sine and cosine functions at multiple frequencies
- **Rotary Position Embedding (RoPE)**: Applying pairwise rotations to hidden dimensions so that interactions between queries and keys can reflect relative position offsets
- **Tensor Operations**: Understanding `view` / `reshape`, `transpose` / `permute`, `squeeze` / `unsqueeze`, and the role of `gather` / `scatter`
- **Broadcasting**: Using PyTorch broadcasting rules to write compact tensor code, from simple scalar cases to attention masks and head-specific biases in Transformers

### Experiment Further

- Visualize sinusoidal positional encodings with `matplotlib` to better see the frequency patterns across dimensions
- Compare how different tokenizer families split the same text (for example, GPT-2, LLaMA, and Qwen tokenizers)
- Change the RoPE base frequency and observe how it changes the relative-position pattern of rotated dot products
- Estimate the parameter and memory cost of embedding tables for different vocabulary sizes and hidden dimensions
- Build both padding masks and causal masks with broadcasting, and inspect how their shapes align with attention score tensors


---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
